# 04 · Retention & Churn

**Questions**
1. Is a 14-day inactivity threshold a sound definition of churn?
2. **When** do users drop off, and do newer signup cohorts retain better?
3. **Which segments** (channel, platform, experience) retain best?
4. Which **week-1 behaviors** go with D7 retention? Is there a "magic number"?
5. Which factors still matter **after controlling for the others**?

**Caveat for the whole notebook:** this is observational data. Results describe **associations**. Motivation drives both engagement and retention, so no finding here proves that a behavior *causes* retention. Causal claims come from the A/B test (Step 6).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test

pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
PROC = Path("processed"); REPORTS = Path("reports"); REPORTS.mkdir(exist_ok=True)

F = pd.read_parquet(PROC / "user_features.parquet")
for c in ["retained_d7", "retained_d28"]:
    F[c] = F[c].astype(float)              # True/False with NaN where the window isn't observable
events = pd.read_parquet(PROC / "events.parquet", columns=["user_id", "event_name", "ts", "in_ios_tracking_gap"])
users = pd.read_parquet(PROC / "users.parquet", columns=["user_id", "signup_ts", "timezone"])
print(F.shape)

In [ ]:
# Practice-activity days (same definition as Step 3: local calendar day, completes + iOS-gap starts)
act = events[(events.event_name == "practice_complete") |
             ((events.event_name == "practice_start") & events.in_ios_tracking_gap)].merge(users, on="user_id")
parts = []
for zone, idx in act.groupby("timezone").groups.items():
    loc = act.loc[idx]
    day = (loc.ts.dt.tz_convert(zone).dt.tz_localize(None).dt.normalize()
           - loc.signup_ts.dt.tz_convert(zone).dt.tz_localize(None).dt.normalize()).dt.days
    parts.append(day)
act["day"] = pd.concat(parts)
act = act[["user_id", "day"]].drop_duplicates().sort_values(["user_id", "day"])
print(f"practice user-days: {len(act):,}")

---
## 1. Validate the churn threshold
Among users who **did** come back, how long were they away? And once someone has been inactive for *k* days, how likely are they to return?

In [ ]:
act["gap"] = act.groupby("user_id").day.diff()
gaps = act.gap.dropna()
print("Gap between practice days (returning users):")
print(gaps.describe(percentiles=[.5, .75, .9, .95, .99]).round(1).to_string())

obs = F.set_index("user_id").obs_days
last = F.set_index("user_id").last_practice_day.dropna()
tail = obs.reindex(last.index) - last
rows = []
for k in [3, 5, 7, 10, 14, 21, 30]:
    came_back = int((gaps > k).sum()); still_gone = int((tail > k).sum())
    rows.append(dict(inactive_more_than=k, came_back=came_back, never_returned=still_gone,
                     p_return=round(came_back / (came_back + still_gone), 3)))
thr = pd.DataFrame(rows); thr

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thr.inactive_more_than, thr.p_return * 100, marker="o")
ax.axvline(14, ls="--", c="grey"); ax.text(14.5, 40, "14-day threshold", color="grey")
ax.set_xlabel("Days inactive"); ax.set_ylabel("% who ever return"); ax.set_title("Probability of returning after k inactive days")
plt.tight_layout(); plt.savefig(REPORTS / "churn_threshold.png", dpi=120); plt.show()

---
## 2. Cohort retention
Users are grouped by **signup week**, and the x-axis is **weeks since signup**, so new signups can't hide churn. A cell is only filled once the whole cohort has had the full week.

In [ ]:
F["signup_week"] = F.signup_ts.dt.tz_convert(None).dt.to_period("W").dt.start_time
act_w = act.assign(week=act.day // 7).drop_duplicates(["user_id", "week"])
act_w = act_w.merge(F[["user_id", "signup_week", "obs_days"]], on="user_id")
size = F.groupby("signup_week").size()
min_obs = F.groupby("signup_week").obs_days.min()

ret = act_w.groupby(["signup_week", "week"]).user_id.nunique().unstack(fill_value=0)
ret = ret.div(size, axis=0)
for w in ret.columns:                                   # blank cells the cohort hasn't fully reached
    ret.loc[min_obs < (w + 1) * 7 - 1, w] = np.nan
ret = ret.loc[:, :12]
ret.index = ret.index.strftime("%b %d")
(ret * 100).round(0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1.3, 1]})
im = axes[0].imshow(ret.values * 100, aspect="auto", cmap="Blues", vmin=0, vmax=60)
axes[0].set_yticks(range(len(ret))); axes[0].set_yticklabels(ret.index, fontsize=7)
axes[0].set_xticks(range(ret.shape[1])); axes[0].set_xticklabels([f"W{w}" for w in ret.columns])
axes[0].set_xlabel("Weeks since signup (W0 = signup week)"); axes[0].set_title("% of cohort practicing, by signup week")
fig.colorbar(im, ax=axes[0], shrink=.7)

avg = ret.mean() * 100
axes[1].plot(avg.index, avg.values, marker="o")
axes[1].set_xlabel("Weeks since signup"); axes[1].set_ylabel("% practicing"); axes[1].set_title("Average retention curve (all cohorts)")
axes[1].set_ylim(0, 100)
plt.tight_layout(); plt.savefig(REPORTS / "cohort_retention.png", dpi=120); plt.show()
print((avg).round(1).to_string())

In [ ]:
# Is a weak cohort a product problem or a mix problem? Compare acquisition mix by signup week
mix = pd.crosstab(F.signup_week, F.acquisition_channel, normalize="index").round(2)
mix.index = mix.index.strftime("%b %d")
mix.assign(w1_retention=(ret[1]).round(2))

---
## 3. Survival curves by segment (Kaplan–Meier)
**Duration** = days until the last practice before churn (`survival_days`). **Event** = churned (no practice in the final 14 observed days). Users still active are **censored**, meaning "lasted at least this long". A multivariate log-rank test checks whether the curves differ.

In [ ]:
def km_plot(col, ax, order=None, min_n=200):
    groups = order or F[col].value_counts().index.tolist()
    sub = F[F[col].isin(groups)]
    for g in groups:
        d = sub[sub[col] == g]
        if len(d) < min_n: continue
        KaplanMeierFitter().fit(d.survival_days, d.churned, label=f"{g} (n={len(d):,})").plot_survival_function(ax=ax, ci_show=False)
    p = multivariate_logrank_test(sub.survival_days, sub[col], sub.churned).p_value
    ax.set_title(f"{col}  (log-rank p = {p:.1e})"); ax.set_xlim(0, 120); ax.set_ylim(0, 1)
    ax.set_xlabel("days since signup"); ax.set_ylabel("share not yet churned")

F["practice_days_w1_bucket"] = pd.cut(F.practice_days_w1, [-1, 0, 1, 2, 3, 7], labels=["0", "1", "2", "3", "4+"]).astype(str)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
km_plot("practice_days_w1_bucket", axes[0, 0], order=["0", "1", "2", "3", "4+"])
km_plot("acquisition_channel", axes[0, 1])
km_plot("platform", axes[1, 0])
km_plot("meditation_experience", axes[1, 1], order=["none", "some", "regular", "not_answered"])
plt.tight_layout(); plt.savefig(REPORTS / "km_segments.png", dpi=120); plt.show()

In [ ]:
# Median survival and D7 / D28 retention by segment
def seg_table(col):
    g = F.groupby(col)
    return pd.DataFrame({"users": g.size(), "d7_retention_%": (g.retained_d7.mean() * 100).round(1),
                         "d28_retention_%": (g.retained_d28.mean() * 100).round(1)}).sort_values("d7_retention_%", ascending=False)
for col in ["acquisition_channel", "platform", "meditation_experience", "age_band"]:
    print(f"\n{col}"); print(seg_table(col).to_string())

---
## 4. Week-1 behaviors and D7 retention
All features come from **days 0–6**, so there's no overlap with the D7 window (days 7–13).

In [ ]:
def d7_by(col, order=None):
    t = F.groupby(col).agg(users=("retained_d7", "size"), d7_retention=("retained_d7", "mean"))
    t["d7_retention"] = (t.d7_retention * 100).round(1)
    return t.reindex(order) if order else t

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
specs = [("practice_days_w1", None), ("longest_streak_w1", None),
         ("first_practice_bucket_w1", ["same_session", "same_day", "day_1", "days_2_6", "never_in_w1"]),
         ("preferred_time_w1", ["morning", "midday", "evening", "mixed", "none"]),
         ("set_reminder", None), ("first_practice_type", ["lesson", "seated_meditation", "active_meditation", "none"])]
for ax, (col, order) in zip(axes.flat, specs):
    t = d7_by(col, order)
    ax.bar(t.index.astype(str), t.d7_retention, color="#4C78A8")
    for i, (v, n) in enumerate(zip(t.d7_retention, t.users)):
        ax.text(i, v + 1, f"{v:.0f}%\n(n={n:,})", ha="center", fontsize=7)
    ax.set_title(col); ax.set_ylim(0, 110); ax.set_ylabel("D7 retention %"); ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig(REPORTS / "d7_by_week1_behavior.png", dpi=120); plt.show()
d7_by("practice_days_w1")

In [ ]:
# The "magic number": the biggest jump in D7 retention between consecutive practice-day counts
t = d7_by("practice_days_w1")
t["jump_pp"] = t.d7_retention.diff().round(1)
t

---
## 5. Multivariable models
What still predicts retention **with the other factors held fixed**?
- **Logistic regression** for `retained_d7` (week-1 features only), reported as odds ratios.
- **Cox proportional hazards** for time to churn, reported as hazard ratios (HR < 1 = lower churn risk).

Experiment variant is included as a control, since treatment users had a different onboarding.

In [ ]:
M = F.copy()
M["log_minutes_w1"] = np.log1p(M.minutes_w1)
M["morning"] = (M.preferred_time_w1 == "morning").astype(int)
M["short_first_practice"] = (M.first_practice_length_min <= 10).astype(int)
M["set_reminder"] = M.set_reminder.astype(int)
M["age_55_plus"] = M.age_band.isin(["55-64", "65+"]).astype(int)
M["variant"] = M.variant.fillna("not_in_experiment")
M["retained_d7"] = M.retained_d7.astype(float)

formula = ("retained_d7 ~ practice_days_w1 + log_minutes_w1 + set_reminder + morning + short_first_practice"
           " + C(acquisition_channel, Treatment('organic')) + C(platform, Treatment('ios'))"
           " + C(meditation_experience, Treatment('none')) + age_55_plus + C(variant, Treatment('not_in_experiment'))")
logit = smf.logit(formula, data=M).fit(disp=False)
odds = pd.DataFrame({"odds_ratio": np.exp(logit.params), "ci_low": np.exp(logit.conf_int()[0]),
                     "ci_high": np.exp(logit.conf_int()[1]), "p": logit.pvalues}).round(3)
odds.index = odds.index.str.replace(r"C\((\w+), Treatment\('[\w_]+'\)\)\[T\.", r"\1=", regex=True).str.rstrip("]")
print(f"pseudo R2 = {logit.prsquared:.3f}, n = {int(logit.nobs):,}")
odds.drop("Intercept")

In [ ]:
cox_cols = ["survival_days", "churned", "set_reminder", "morning", "short_first_practice", "age_55_plus"]
X = M[cox_cols].copy()
X["churned"] = X.churned.astype(int)
dums = pd.get_dummies(M[["acquisition_channel", "platform", "meditation_experience"]], drop_first=False).astype(int)
dums = dums.drop(columns=["acquisition_channel_organic", "platform_ios", "meditation_experience_none"])
X = X.join(dums)
X["survival_days"] = X.survival_days.clip(lower=0.5)       # Cox needs positive durations
cph = CoxPHFitter(penalizer=0.01).fit(X, duration_col="survival_days", event_col="churned")
hr = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].round(3)
hr.columns = ["hazard_ratio", "ci_low", "ci_high", "p"]
hr.sort_values("hazard_ratio")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
o = odds.drop("Intercept").sort_values("odds_ratio")
axes[0].errorbar(o.odds_ratio, range(len(o)), xerr=[o.odds_ratio - o.ci_low, o.ci_high - o.odds_ratio], fmt="o")
axes[0].axvline(1, c="grey", ls="--"); axes[0].set_yticks(range(len(o))); axes[0].set_yticklabels(o.index, fontsize=8)
axes[0].set_xscale("log"); axes[0].set_title("D7 retention: odds ratios (>1 = more likely retained)")
h = hr.sort_values("hazard_ratio")
axes[1].errorbar(h.hazard_ratio, range(len(h)), xerr=[h.hazard_ratio - h.ci_low, h.ci_high - h.hazard_ratio], fmt="o", c="#E45756")
axes[1].axvline(1, c="grey", ls="--"); axes[1].set_yticks(range(len(h))); axes[1].set_yticklabels(h.index, fontsize=8)
axes[1].set_title("Time to churn: hazard ratios (<1 = lower churn risk)")
plt.tight_layout(); plt.savefig(REPORTS / "retention_models.png", dpi=120); plt.show()

---

## 7. Findings

**1. The 14-day churn threshold holds up.** 99% of returning users come back within 10 days. After 14 inactive days, only **1.2%** ever return, compared with 61% after 3 days. The **7–10 day** window (18% → 6% return) is where a re-engagement nudge has the best chance.

**2. The biggest drop is week 1 → week 2.** On average, 84% practice in their signup week, **35% in week 1**, 27% in week 2, then a slow decline to about 10% by week 12. The first week is where most users are won or lost.

**3. Cohorts are stable, with one exception that isn't a product problem.** Retention curves are similar across signup weeks. The **Oct 5 cohort** is weakest (30% in week 1, 13% in week 4), but **55% of it came from paid social** (vs ~22% normally) during the October ad burst. It's a **mix effect**: the product didn't get worse, the audience changed. Always check the acquisition mix before blaming the product.

**4. The second practice day is the "magic number."** D7 retention by week-1 practice days: 5% (0 days) → 11% (1) → **41% (2)** → **65% (3)** → 79% (4) → 94% (7). The biggest jumps come from **getting a user to practice on a second and third day** (+29 and +24 points). One practice day is barely better than none.

**5. Acquisition channel is the strongest profile signal.** D7 retention: partner organizations 49% · referral 43% · organic 37% · app store search 34% · **paid social 23%**. With other factors held fixed, paid social still carries **47% higher churn risk** (HR 1.47) and partner organizations **30% lower** (HR 0.70). Web users churn more than app users (HR 1.25). Meditation experience matters little.

**6. Early habits that still matter after controls:**

| Behavior | D7 odds ratio | Churn hazard ratio |
|---|---|---|
| Each extra practice day in week 1 | **2.61** | — |
| Mostly morning practice | 1.25 | **0.66** |
| Short (≤10 min) first practice | 1.30 | 0.75 |
| Set a reminder | 1.17 | 0.78 |
| Age 55+ | 1.22 | 0.90 |

Minutes add little once practice days are known (OR 1.11 per log-minute): **frequency matters more than volume.**

**7. Missing age isn't informative here.** Users with `age_band = unknown` retain at the average rate (35.7%), so keeping them as a category rather than imputing cost nothing.

### Caveats
- **These are associations, not causes.** Motivated users practice more often, in the morning, and set reminders, *and* they stay. The raw reminder gap (42% vs 32%) shrinks to an odds ratio of 1.17 once practice days are controlled. The true causal effect could be smaller still.
- The model controls for what's observed. **Motivation itself is unobserved**, so it can't be fully removed.
- **What would establish causality:** randomized tests, for example a "suggest a short first practice" test, or randomized reminder prompts. The onboarding A/B test (Step 6) is the first such test.

### Recommendations (to be validated by experiment)
1. **Focus on the second practice day.** Design day 1–3 nudges to bring users back for a second session.
2. **Default first-time users to short (≤10 min) content.**
3. **Suggest morning practice** in onboarding, and test it.
4. **Review paid social spend:** its users retain at less than half the rate of partner-organization users. Combine with CAC (Step 7) to compare cost per *retained* user.
5. **Trigger re-engagement on days 7–10 of inactivity**, before the 14-day point of no return.